# Milestone 3 on Kaggle: LFCC + LCNN, and the In-the-Wild test set

Settings: **Accelerator = GPU (T4 or P100)**, **Internet = on**. Run with *Save Version, Save & Run All*.

What it does:
1. downloads ASVspoof 2019 LA (ODC-By, doi:10.7488/ds/2555) and In-the-Wild (Muller et al. 2022, Hugging Face `mueller91/In-The-Wild`, CC-BY-SA 4.0);
2. extracts LFCC frames for train, dev, eval and In-the-Wild;
3. extracts pooled wav2vec 2.0 features for In-the-Wild (train/dev/eval were done in milestone 2);
4. scores In-the-Wild with the milestone 1 GMMs;
5. trains the LCNN with 3 seeds and scores eval + In-the-Wild.

Outputs, in `/kaggle/working`: `w2v2_base/itw*` (about 1.2 GB) and `results/` (score files, a few MB).

In [ ]:
import torch, torchaudio, shutil, os
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count())
print('free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/audio-deepfake-generalization.git /tmp/repo
!pip install -q soundfile
%cd /tmp/repo

In [ ]:
!python scripts/download_asvspoof2019.py --out /tmp/data

In [ ]:
# prints the metadata header, class counts and sample rates: check these before trusting anything below
!python scripts/download_in_the_wild.py --out /tmp/itw

In [ ]:
!python scripts/extract_lfcc.py --la-root /tmp/data/LA --itw-root /tmp/itw --feat-root /tmp/features --partitions train dev eval itw

In [ ]:
!python scripts/extract_wav2vec2.py --itw-root /tmp/itw --partitions itw --out /kaggle/working/w2v2_base --device cuda

In [ ]:
!python scripts/score_gmm.py --gmm-dir results/lfcc_gmm_asvspoof2019_b02 --feat-dir /tmp/features/lfcc_asvspoof2019_b02/itw --out /kaggle/working/results/lfcc_gmm_asvspoof2019_b02/scores_itw.csv

In [ ]:
!python scripts/train_lcnn.py --feat-dir /tmp/features/lfcc_asvspoof2019_b02 --la-root /tmp/data/LA --itw-root /tmp/itw --out /kaggle/working/results/lfcc_lcnn --device cuda

In [ ]:
# label table (utt_id, attack, key, label) so the laptop can evaluate In-the-Wild without the audio
import sys; sys.path.insert(0, '/tmp/repo/src')
from adg.datasets import export_itw_labels
print(export_itw_labels('/tmp/itw', '/kaggle/working/itw_labels.csv'), 'labels written')
!ls -laR /kaggle/working | head -50